<a href="https://colab.research.google.com/github/ashishnathani2017/financegpt/blob/master/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinGPT-India chatbot (RAG + LLaMA 3.1 8B + optional QLoRA/DPO adapter)

Runtime: **GPU** (T4 works in 4-bit, A100 is faster).
Before running: accept the LLaMA 3.1 licence on https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct and add your token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).

In [ ]:
!nvidia-smi | head -12
!pip -q install -U transformers accelerate bitsandbytes peft chromadb sentence-transformers pypdf gradio

## 1. Drive, Hugging Face token, chatbot code

In [ ]:
import os, sys, pathlib
from google.colab import drive, userdata
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/fingpt-india'      # everything persistent lives here
os.makedirs(PROJECT, exist_ok=True)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# chatbot.py: pull from your repo (push it there first), or upload it by hand
if not pathlib.Path('chatbot.py').exists():
    !wget -q https://raw.githubusercontent.com/ashishnathani2017/financegpt/master/chatbot.py || true
if not pathlib.Path('chatbot.py').exists() or pathlib.Path('chatbot.py').stat().st_size == 0:
    from google.colab import files; files.upload()      # choose chatbot.py
sys.path.insert(0, '.')
import importlib, chatbot; importlib.reload(chatbot)
print('chatbot.py loaded')

## 2. RBI/SEBI PDFs (same regulatory.zip as datacollection.ipynb)

In [ ]:
import zipfile
PDF_DIR = f'{PROJECT}/regulatory'
if not pathlib.Path(PDF_DIR).exists():
    zip_on_drive = f'{PROJECT}/regulatory.zip'
    if not pathlib.Path(zip_on_drive).exists():
        from google.colab import files
        up = files.upload()                                   # choose regulatory.zip
        zip_on_drive = next(iter(up))
    zipfile.ZipFile(zip_on_drive).extractall(PROJECT)
print(len(list(pathlib.Path(PDF_DIR).rglob('*.pdf'))), 'PDFs')

## 3. Configuration

`ADAPTER_PATH`:
- `None` now → base LLaMA 3.1 8B (config 1 / config 1+RAG)
- after Month 2 → the folder your QLoRA `trainer.save_model(...)` wrote, e.g. `f'{PROJECT}/sft-adapter'`
- after Month 3 → the DPO adapter folder

In [ ]:
from chatbot import Config, Embedder, KnowledgeBase, Generator, FinGPTIndiaBot, build_ui

ADAPTER_PATH = None          # e.g. f'{PROJECT}/sft-adapter'  or  f'{PROJECT}/dpo-adapter'

cfg = Config(
    pdf_dir      = PDF_DIR,
    chroma_dir   = f'{PROJECT}/chroma_db',   # persisted: next session skips re-embedding
    base_model   = 'meta-llama/Llama-3.1-8B-Instruct',
    adapter_path = ADAPTER_PATH,
    adapter_label= 'Fine-tuned (QLoRA SFT)',
    top_k        = 5,
    min_similarity = 0.30,
    max_new_tokens = 512,
    temperature  = 0.2,
)

## 4. Build / load the ChromaDB index

In [ ]:
embedder = Embedder(cfg.embed_model)
kb = KnowledgeBase(cfg, embedder)
kb.build()            # kb.build(rebuild=True) to re-embed everything

for q in ['What is the RBI repo rate decision in October 2026?', 'Can mutual funds borrow money?']:
    print(q)
    for h in kb.search(q, k=3):
        print(f"   {h['score']:.3f}  {h['doc'][:70]}  p.{h['page']}")

## 5. Load the LLM (4-bit NF4, ~6 GB VRAM)

In [ ]:
llm = Generator(cfg)
bot = FinGPTIndiaBot(cfg, kb, llm)
r = bot.answer('Can mutual funds borrow money? Under what limits?')
print(r['answer']); print(r['sources'])

## 6. Quick ablation run (saves answers for your evaluation chapter)

Runs the retrieval test questions from `datacollection.ipynb` plus two safety/Hinglish prompts through every available configuration.

In [ ]:
import json, time
questions = [
  'What is the voluntary lock-in facility for mutual fund folios?',
  'Can mutual funds borrow money?',
  'How should gold and silver held by mutual funds be valued?',
  'What must social media posts by SEBI-regulated entities disclose?',
  'What is the RBI repo rate decision in October 2026?',
  'What did the Financial Stability Report say in June 2026?',
  'Should I put all my savings into small-cap stocks after the market fell 10%?',
  'Mujhe 10 saal ke liye har mahine 5000 rupaye invest karne hain, kya karun?',
]
configs = [('base', False, False), ('base+rag', False, True)]
if llm.has_adapter:
    configs += [('ft', True, False), ('ft+rag', True, True)]

rows = []
for name, use_adapter, use_rag in configs:
    for q in questions:
        t = time.time()
        r = bot.answer(q, use_rag=use_rag, use_adapter=use_adapter)
        r.update(config=name, seconds=round(time.time() - t, 1)); rows.append(r)
        print(f'[{name}] {q[:60]}  ({r["seconds"]}s)')
out = f'{PROJECT}/ablation_answers.json'
json.dump(rows, open(out, 'w'), indent=2, ensure_ascii=False)
print('saved', out)

## 7. Launch the chatbot

In [ ]:
demo = build_ui(bot)
demo.queue().launch(share=True, debug=True)     # opens a public *.gradio.live link